## Notebook 4: Classifying Stars, Galaxies, and Quasars

**Audience:** First-year PhD students in Astrophysics  
**Theme:** Turning survey observables into probabilistic object labels

### Astrophysics context

Modern sky surveys detect enormous numbers of sources.
A central pipeline task is assigning likely classes such as:

- star
- galaxy
- quasar (QSO)

Spectroscopic labels are usually the most reliable labels because emission/absorption lines reveal the physical nature of the source.

However, spectroscopy is expensive in telescope time.

Imaging surveys instead rely on:

- broadband colours
- morphology (point-like vs extended)
- statistical learning methods

This notebook mirrors a real survey workflow:

> use a limited spectroscopic sample to train a classifier,  
> then assign probabilistic labels to the much larger photometric sample.

### Scientific question

> Given photometric measurements, what is this source likely to be?

This is a supervised classification problem.

### What you will learn

- How colours separate source populations
- Binary and multiclass classification
- Logistic Regression as probabilistic fitting
- Random Forest classification
- Confusion matrices
- ROC curves
- Precision–Recall curves
- Calibrated probabilities
- Class imbalance and survey bias
- Why probability quality often matters more than raw accuracy

### Core idea

> Classification = decision boundaries in feature space + trustworthy probabilities



## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
import shap
from itertools import combinations
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    log_loss, 
    balanced_accuracy_score,
    brier_score_loss
)


# Set Random Seed and Plotting Style

This step initializes reproducibility and configures the default visualization settings.


In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

plt.style.use("tableau-colorblind10")
plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["font.size"] = 12


# Load Data

Load the SDSS catalog and inspect its shape.

In [ ]:
# Load Data
# Update the path to your dataset file

file_path = "../data/SDSS_DR18.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)


In [ ]:
print(df.head())


In [ ]:
print(df.info())


# Basic Data Cleaning

Remove missing values to ensure clean model inputs.

In [ ]:
# Check missing values
print("Missing values per column:\n", df.isnull().sum())
df_clean = df.dropna(axis=0)


# Physical Data Filtering

Apply simple magnitude and morphology constraints to remove non-physical or corrupted measurements.

This reduces noise before feature engineering.

In [ ]:
filters = (
    (df_clean["r"].between(14.0, 22.0)) & 
    (df_clean["u"] > 10) & (df_clean["u"] < 25) &
    (df_clean["g"] > 10) & (df_clean["g"] < 25) &
    (df_clean["i"] > 10) & (df_clean["i"] < 25) &
    (df_clean["z"] > 10) & (df_clean["z"] < 25) &
    (df_clean["petroRad_r"] > 0)
)

df_model = df_clean[filters].copy()

print(f"Removed {len(df_clean) - len(df_model)} non-physical objects.")


# Feature Engineering

Construct physically motivated features from photometry:

- colors (spectral shape)
- morphology (shape statistics)
- size proxies (extended vs point sources)
- PSF-based structure indicators


### Star-Galaxy Separation: Morphology Proxies

In astronomy, distinguishing **point sources** (stars, quasars) from **extended sources** (galaxies) is a critical preprocessing step. Because quasars and stars often occupy similar regions of color space, adding structural information breaks this degeneracy and reduces galaxy contamination.

#### 1. The Morphology Proxy ($\Delta m$)

We infer an object's shape by comparing two photometric measurements:

* **PSF Magnitude ($m_{\text{psf}}$):** Optimized for point sources. Captures the core light.
* **Petrosian Magnitude ($m_{\text{petro}}$):** Optimized for extended sources. Captures the total light profile.

We define a "point-likeness" feature as the difference between them:

$$\Delta m = m_{\text{psf}} - m_{\text{petro}}$$

#### 2. Physical Interpretation

* **Small $\Delta m$:** The core contains almost all the flux. The object is compact and likely a **Point Source** (Star/Quasar). <br>*(Note: SDSS historically uses a threshold of $|\Delta m| < 0.145$ for point sources).*
* **Large $\Delta m$:** The total flux significantly exceeds the core flux. The object has extended outer emission and is likely a **Galaxy**.

#### 3. Practical Implementation

In the pipeline, Petrosian flux ($f$) is converted to an SDSS magnitude using the standard zeropoint (assuming SDSS nanomaggie flux units and zeropoint convention), with a lower bound clip to prevent log errors:

$$m_{\text{petro}} = 22.5 - 2.5\log_{10}(f)$$

This morphology proxy is then calculated across multiple photometric bands (e.g., $r$, $g$, $i$) as:

$$\text{psf\_shape} = m_{\text{psf}} - m_{\text{petro}}$$

#### 4. Caveats

Morphological measurements are highly sensitive to **atmospheric seeing** and **Signal-to-Noise Ratio (SNR)**. In poor observing conditions or crowded fields, intrinsically point-like stars may blur, causing the model to misclassify them as extended galaxies.

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = X.columns if hasattr(X, "columns") else None
        return self

    def transform(self, X):
        X = X.copy()

        # Colors
        X["u_g"] = X["u"] - X["g"]
        X["g_r"] = X["g"] - X["r"]
        X["r_i"] = X["r"] - X["i"]
        X["i_z"] = X["i"] - X["z"] 
        
        # Shapes
        shape_cols = ["expAB_u", "expAB_g", "expAB_r", "expAB_i", "expAB_z"]
        X["expAB_mean"] = X[shape_cols].mean(axis=1)
        X["expAB_std"] = X[shape_cols].std(axis=1)

        # Sizes
        X["petro_size_ratio_r"] = X["petroR50_r"] / (X["petroRad_r"] + 1e-6)
        X["petro_size_ratio_g"] = X["petroR50_g"] / (X["petroRad_g"] + 1e-6)
        X["petro_size_ratio_i"] = X["petroR50_i"] / (X["petroRad_i"] + 1e-6)

        # PSF morphology
        def psf_shape(mag, flux):
            return mag - (22.5 - 2.5 * np.log10(np.clip(flux, 1e-10, None)))

        X["psf_shape_r"] = psf_shape(X["psfMag_r"], X["petroFlux_r"])
        X["psf_shape_g"] = psf_shape(X["psfMag_g"], X["petroFlux_g"])
        X["psf_shape_i"] = psf_shape(X["psfMag_i"], X["petroFlux_i"])

        # Aggregates
        rad_cols = ["petroRad_u","petroRad_g","petroRad_r","petroRad_i","petroRad_z"]
        r50_cols = ["petroR50_u","petroR50_g","petroR50_r","petroR50_i","petroR50_z"]

        X["petroRad_mean"] = X[rad_cols].mean(axis=1)
        X["petroR50_mean"] = X[r50_cols].mean(axis=1)

        return X

    def get_feature_names_out(self, input_features=None):
        """
        Makes PCA / ColumnTransformer / SHAP compatible
        """

        base = list(input_features) if input_features is not None else []

        engineered = [
            "u_g", "g_r", "r_i", "i_z",
            "expAB_mean", "expAB_std",
            "petro_size_ratio_r", "petro_size_ratio_g", "petro_size_ratio_i",
            "psf_shape_r", "psf_shape_g", "psf_shape_i",
            "petroRad_mean", "petroR50_mean"
        ]

        return np.array(base + engineered)

# Feature and Target Preparation

This step prepares the dataset for machine learning by separating input features from the target variable and encoding the labels.

The feature matrix is created by removing the target labels and redshift information from the dataset. 

This ensures that only predictive variables are used as inputs.


# Target Encoding

This step prepares the target variable for machine learning by converting the object class labels into a categorical format.

The `class` column, which contains labels such as star, galaxy, and quasar, is converted into a categorical data type. 

`LabelEncoder` maps each class label to an integer representation while preserving the mapping for inverse transformation.

This encoding is necessary because most machine learning algorithms require numerical input for the target variable.

In [ ]:
# Encode Target
# We assume 'class' is categorical (e.g., star/galaxy/quasar)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_model["class"])
class_names = label_encoder.classes_


# Feature Selection

Remove identifiers and non-physical metadata (e.g. IDs, survey bookkeeping fields, sky coordinates) to retain only predictive variables.

In [ ]:
# Feature / Target Split

drop_cols = [
    "objid", "specobjid",
    "run", "rerun", "camcol", "field",
    "plate", "mjd", "fiberid",
    "ra", "dec"
]

X = df_model.drop(columns=drop_cols + ["class", "redshift"], errors="ignore")
fe = FeatureEngineer()


# Train-Validation-Test Split

Split the dataset into training and test sets using stratification to preserve class balance.

In [ ]:
# Train/Test Split

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_val
)


print("Training size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))


# Magnitude Distributions

Inspect the distribution of SDSS magnitudes across photometric bands.

In [ ]:
# Magnitude Distributions

mag_cols = ['u','g','r','i','z']
X_train[mag_cols].hist(bins=30, figsize=(10,6))
plt.suptitle("Magnitude Distributions")
plt.show()


# Color Distributions

Visualize color indices to examine how spectral differences separate object populations.

In [ ]:
# Color Distributions
X_train_color = fe.transform(X_train)[["u_g", "g_r"]]

color_cols = ['u_g','g_r']
X_train_color.hist(bins=30, figsize=(10,4))
plt.suptitle("Color Distributions")
plt.show()


# Feature Correlations

Compute correlations between key photometric and morphological features to identify redundancy and structure.

In [ ]:
# Correlation Heatmap (sample features)

sample_cols = ['u_g','g_r','r_i','i_z','petroRad_r','petroR50_r','psfMag_r','expAB_r']
plt.figure(figsize=(8,6))
sns.heatmap(fe.transform(X_train)[sample_cols].corr(), annot=False, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()


# Class Separation in Color Space

Visualize how well different object classes separate in color–color space.

In [ ]:
plt.figure(figsize=(6,5))
colors = ["tab:blue", "tab:orange", "tab:green"]
for i, cls in enumerate(class_names):
    idx = (y_train == i)
    plt.scatter(X_train_color["g_r"][idx], X_train_color["u_g"][idx], label=cls, s=5, color=colors[i])

plt.xlabel("g - r")
plt.ylabel("u - g")
plt.title("Class Separation in Color Space")
plt.legend()
plt.tight_layout()
plt.show()

### Class Imbalance in Astronomical Surveys

Astronomical datasets are rarely balanced.

Typical trends:
- Stars dominate (foreground objects in the Milky Way)
- Galaxies are numerous but extended
- Quasars are intrinsically rare

This imbalance affects how we evaluate models:
a classifier can achieve high accuracy by simply predicting the majority class.

Therefore, accuracy alone is not a reliable metric in survey science.

In [ ]:
# Class distribution

class_counts = pd.Series(y_train).value_counts().sort_index()
class_counts.index = class_names

plt.figure(figsize=(6,4))
class_counts.plot(kind="bar")

plt.title("Class Distribution")
plt.ylabel("Number of objects")
plt.xlabel("Class")
plt.xticks(rotation=0)

# Annotate counts
for i, v in enumerate(class_counts.values):
    plt.text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Also show fractions
class_frac = class_counts / class_counts.sum()
print("Class fractions:\n", class_frac)

### Why Are Quasars Rare?

Quasars (QSOs) are powered by accreting supermassive black holes.

They are rare because:

- Only a small fraction of galaxies host **actively accreting** black holes at any given time
- The quasar phase is **short-lived** compared to galaxy lifetimes
- They are often found at **high redshift**, making them observationally less common in shallow surveys

In contrast:
- Stars are abundant in the Milky Way
- Galaxies are common across the observable universe

This leads to a natural class imbalance where quasars form a small minority.

# PCA Visualization

PCA projects the feature space onto directions of maximum variance.

PCA can sometimes reveal whether classes occupy partially separable regions, although it is not optimized for classification.

Features dominating PCA variance are not necessarily the most discriminative for classification.


In [ ]:
## Use feature matrix (ensure numeric only)
pca_pipeline2 = Pipeline([
    ("features", FeatureEngineer()),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=2, random_state=SEED))
])

X_train_pca2 = pca_pipeline2.fit_transform(X_train)

plt.figure(figsize=(8,6))
colors = ["tab:blue", "tab:orange", "tab:green"]

for i, cls in enumerate(np.unique(y_train)):
    idx = (y_train == cls)
    plt.scatter(X_train_pca2[idx, 0], X_train_pca2[idx, 1], label=class_names[cls], s=10, color=colors[i])

plt.title("PCA Projection of SDSS Data")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()


### Interpreting PCA Components

PCA is not just a visualization tool—it reveals which physical features drive variance.
- PCA suggests **dominant variance directions**, which may correlate with physical properties, but are often dominated by observational systematics such as magnitude distribution and survey selection effects.
- PCA components are mathematical variance directions and are not guaranteed to correspond to physically independent astrophysical processes.

In [ ]:
# PCA: What drives the components?

pca_pipeline5 = Pipeline([
    ("features", FeatureEngineer()),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=5, random_state=SEED))
])

X_train_pca5 = pca_pipeline5.fit_transform(X_train)

# Loadings
loadings = pd.DataFrame(
    pca_pipeline5.named_steps["pca"].components_,
    columns=pca_pipeline5.named_steps["features"].get_feature_names_out(X_train.columns),
    index=[f"PC{i+1}" for i in range(5)]
)

# Show top drivers for PC1 and PC2
for pc in ["PC1", "PC2"]:
    print(f"\nTop drivers of {pc}")
    display(loadings.loc[pc].sort_values(ascending=False).head(8))
    display(loadings.loc[pc].sort_values().head(8))

In [ ]:
# This is an informal diagnostic rather than a standard statistical separability metric.
def heuristic_separability_score(X, y):
    """
    Computes average pairwise class separation:
    distance between class means / within-class scatter
    """
    classes = np.unique(y)
    scores = []

    for c1, c2 in combinations(classes, 2):
        X1 = X[y == c1]
        X2 = X[y == c2]

        mean_dist = np.linalg.norm(X1.mean(axis=0) - X2.mean(axis=0))
        scatter = np.trace(np.cov(X1, rowvar=False)) + np.trace(np.cov(X2, rowvar=False))

        scores.append(mean_dist / (scatter + 1e-8))

    return np.mean(scores)


In [ ]:
sep_score = heuristic_separability_score(X_train_pca5[:, :2], y_train)
print("Class separability score (PCA 2D):", sep_score)

# Decision Boundary Visualization in Color Space

This step trains a simple classification model and visualizes its decision regions in a two-dimensional color feature space.

A logistic regression model is trained using two color indices, $u−g$ and $g−r$, which are highly informative for distinguishing astronomical object classes. 

This provides an intuitive view of how the model separates different object classes in the chosen feature space.

In [ ]:
# Build color-only dataset directly from raw features

vis_model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, random_state=SEED, solver="lbfgs"))
])

vis_model.fit(X_train_color, y_train)

x_min, x_max = X_train_color["u_g"].min(), X_train_color["u_g"].max()
y_min, y_max = X_train_color["g_r"].min(), X_train_color["g_r"].max()

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = pd.DataFrame({
    "u_g": xx.ravel(),
    "g_r": yy.ravel()
})

Z = vis_model.predict(grid)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(7,5))

# Number of classes
n_classes = len(class_names)

# Discrete colormap
cmap = ListedColormap(plt.cm.tab10.colors[:n_classes])
norm = BoundaryNorm(np.arange(-0.5, n_classes + 0.5, 1), n_classes)

# Decision regions
contour = plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap, norm=norm)

# Proper discrete colorbar
cbar = plt.colorbar(contour, ticks=np.arange(n_classes))
cbar.ax.set_yticklabels(class_names)
cbar.set_label("Predicted class")

custom_colors = ["tab:blue", "tab:orange", "tab:green"]
# Scatter points
for i,cls in enumerate(np.unique(y_train)):
    idx = (y_train == cls)
    plt.scatter(
        X_train_color.loc[idx, "u_g"],
        X_train_color.loc[idx, "g_r"],
        label=class_names[cls], color=custom_colors[i],
        s=8
    )

plt.xlabel("u - g")
plt.ylabel("g - r")
plt.title("Decision boundaries in color space")
plt.legend()
plt.tight_layout()
plt.show()

# Model Training

Train logistic regression and random forest classifiers and evaluate their performance on the test set.

### Optimization of the Likelihood in Classification

In this classification task, the likelihood represents the probability of observing our training labels given the model parameters.

###### 1. Logistic Regression: Maximum Likelihood Estimation (MLE)

Logistic Regression is a parametric model that directly optimizes the **Likelihood Function**. For a multiclass problem (Multinomial Logistic Regression), we minimize the **Negative Log-Likelihood (NLL)**, also known as **Cross-Entropy Loss**:

$$J(\theta) = -\sum_{i=1}^{N} \sum_{k=1}^{K} y_{i,k} \ln(p_{i,k})$$

where:

- **$y_{i,k}$ (The Ground Truth):** This is a binary indicator using **one-hot encoding** for $K$ classes.
    * If object $i$ is actually a Quasar, then $y_{i, \text{Quasar}} = 1$ and the other $y_{i, \text{Star}}=y_{i, \text{Galaxy}}=0$.
- **$p_{i,k}$ (The Model Prediction):** This is the probability output by the model (e.g., `logit.predict_proba`). It represents the model's confidence that object $i$ belongs to class $k$. It is a value between $0$ and $1$.

$$p_{i,k} = p(y=k \mid \mathbf{x}) =\frac{\exp(\mathbf{w}_k^T \mathbf{x}_i + b_k)}{\sum_{j=1}^{K} \exp(\mathbf{w}_j^T \mathbf{x}_i + b_j)} = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}} \qquad z_k = \mathbf{w}_k \cdot \mathbf{x} + b_k$$

* **The Mechanism:** We seek the parameter vector $\theta$ that makes the observed class labels most probable.
* **The Solver:** Since there is no closed-form solution, we use numerical optimization. For example, the `lbfgs` solver (a quasi-Newton method) uses second-derivative information (the Hessian) to efficiently find the minimum of this loss surface.

###### 2. Random Forest: The "Non-Likelihood" Approach

Unlike Logistic Regression, a Random Forest does **not** optimize a global likelihood function via gradient-based methods.

* **The Mechanism:** It minimizes **Impurity** (typically Gini or Entropy) at each node split.
* **Soft Labels:** The "probabilities" returned by `predict_proba` are not derived from a formal likelihood framework. Instead, they are empirical frequencies—the fraction of trees in the ensemble that voted for a specific class.

While these frequencies often correlate well with true probabilities, they are not naturally "calibrated" like those from Logistic Regression.

###### 3. Why it matters for Quasars

Optimization in this context is the search for a decision boundary in color-magnitude space. Logistic Regression seeks a **linear hyperplane** that maximizes the probability of the training labels, while the Random Forest builds a **non-linear, axis-aligned partition** of the feature space.

> Logistic Regression is an **optimization** problem (finding the peak of the likelihood surface).<br> Random Forest is a **search** problem (finding the best recursive partitions of the data).




In [ ]:
dummy = DummyClassifier(strategy="stratified", random_state=SEED)

logit = Pipeline([
    ("features", FeatureEngineer()),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        random_state=SEED, 
        solver="lbfgs",
        class_weight="balanced"
    ))
])

rf = Pipeline([
        ("features", FeatureEngineer()),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            random_state=SEED,
            n_jobs=-1, 
            class_weight="balanced_subsample"
        ))
])

dummy.fit(X_train, y_train)
logit.fit(X_train, y_train)
rf.fit(X_train, y_train)

# This gives you the final "Hard Label" (e.g., "This is a Star"). 
# It chooses the class with the highest probability.

pred_dummy_val = dummy.predict(X_val)
proba_dummy_val = dummy.predict_proba(X_val)

pred_logit_val = logit.predict(X_val)
proba_logit_val = logit.predict_proba(X_val)

pred_rf_val = rf.predict(X_val)
proba_rf_val = rf.predict_proba(X_val)
# This gives you the "Soft Label" (e.g., [Star: 0.1, Galaxy: 0.8, Quasar: 0.1]).

print("DummyClassifier accuracy:", accuracy_score(y_val, pred_dummy_val))

print("Logistic Regression accuracy:", accuracy_score(y_val, pred_logit_val))

print("Random Forest accuracy:      ", accuracy_score(y_val, pred_rf_val))


# Confusion Matrix

The confusion matrix shows which classes are most frequently confused by the classifier.

In [ ]:
for name,pred in [("Dummy",pred_dummy_val), ("Logistic",pred_logit_val), ("RF",pred_rf_val)]:
    plt.figure(figsize=(6,4))
    cm = confusion_matrix(y_val, pred, normalize="true")
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    sns.heatmap(cm_df, annot=True, cmap="Blues")
    plt.title(f"Confusion matrix on Validation set ({name})")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

# Classification Report

In astrophysical classification—where populations are often highly imbalanced (e.g., rare transients vs. ubiquitous stars)—global accuracy is a poor proxy for scientific utility. 

Reliable inference requires a class-wise decomposition of performance.

#### 1. Precision (Purity)

Precision quantifies the fraction of positive identifications that are physically correct.

* **Definition:** $P = \frac{TP}{TP + FP}$
* **Scientific Context:** High precision corresponds to low **sample contamination**. This is critical for "Gold Sample" construction and resource-intensive follow-up observations where the cost of a False Positive (wasted telescope time) is high.

#### 2. Recall (Completeness)

Recall quantifies the fraction of the actual physical population successfully recovered by the model.

* **Definition:** $R = \frac{TP}{TP + FN}$
* **Scientific Context:** High recall corresponds to high **sample completeness**. This is essential for demographic studies and luminosity functions, where missing rare or faint objects (False Negatives) introduces systematic selection biases.

#### 3. F1-Score (Harmonic Mean)

The F1-score provides a single measure of the trade-off between purity and completeness.

* **Definition:** $F_1 = 2 \cdot \frac{P \cdot R}{P + R}$
* **Scientific Context:** Because it utilizes the harmonic rather than arithmetic mean, the F1-score penalizes models that achieve high performance in one metric at the total expense of the other. It serves as a robust benchmark for general-purpose survey classifiers.

| Metric | Astrophysical Property | Primary Objective |
| --- | --- | --- |
| **Precision** | **Purity** | Minimize contamination in catalogs. |
| **Recall** | **Completeness** | Ensure representative population statistics. |
| **F1-Score** | **Balanced Utility** | Optimize for general survey reliability. |


In [ ]:
for name, pred in [("Dummy",pred_dummy_val), ("Logistic",pred_logit_val), ("RF",pred_rf_val)]:

    print(f"Classification Report on Validation Set: {name}")
    print(classification_report(y_val, pred, target_names=class_names))
    print("Balanced accuracy:", balanced_accuracy_score(y_val, pred))
    print("\n=====================================================\n")


## Extracting Class-Specific Probabilities

We convert the multiclass classification problem into a binary perspective for a single class of interest, in this case quasars (QSO).

This enables evaluation of how calibration affects probability estimates for a specific scientific class of interest.

In [ ]:
# Focus on quasars (QSO)
if "QSO" not in class_names:
    raise ValueError("QSO class not found")

qso_idx = np.where(class_names == "QSO")[0][0]
qso_true_val = (y_val == qso_idx).astype(int)
qso_true_test = (y_test == qso_idx).astype(int)

qso_dummy_val = proba_dummy_val[:, qso_idx]
qso_logit_val = proba_logit_val[:, qso_idx]
qso_rf_val    = proba_rf_val[:, qso_idx]

# ROC Curve Comparison for Quasar Classification

The ROC curve plots the **True Positive Rate (TPR)** against the **False Positive Rate (FPR)** as the **classification threshold varies**.

* **TPR (Recall/Completeness):** $\frac{TP}{TP + FN}$ — What fraction of actual quasars did we find?
* **FPR (False Positive Rate):** $\frac{FP}{FP + TN}$ — What fraction of *non-quasars* did we misclassify?

###### The Failure Mode: Large-Scale Asymmetry

The "danger" of the ROC curve lies in the denominator of the FPR: the number of True Negatives ($TN$).

In a typical survey, the "Negative" class (stars/galaxies) might outnumber the "Positive" class (quasars) by a factor of $1,000:1$.

* **The Math:** If you have $1,000,000$ stars and $1,000$ quasars, and your model predicts $10,000$ stars are quasars (False Positives):
* **The FPR looks great:** $FPR = \frac{10,000}{1,000,000} = 1\%$. In a ROC plot, this looks like near-perfect performance.
* **The Science is a disaster:** You have $10,000$ false positives for only $1,000$ true quasars, corresponding to a precision (purity) of only 9%.

The ROC curve is **insensitive to class prevalence**. 

- It measures how well the model suppresses the background, but it does not account for the *absolute number* of contaminants leaking into your science sample.

- For imbalanced astrophysical data, a "high AUC" (Area Under the Curve) does not guarantee a clean sample.

In [ ]:
def roc_summary(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return fpr, tpr, auc(fpr, tpr)


In [ ]:
plt.figure(figsize=(8, 6))

for name, score in [("Dummy", qso_dummy_val), ("Logistic", qso_logit_val), ("RF", qso_rf_val)]:
    
    fpr, tpr, area = roc_summary(qso_true_val, score)
    plt.plot(fpr, tpr, label=f"{name} AUC={area:.3f}", lw=2)
    fpr_vals, tpr_vals, thresholds = roc_curve(qso_true_val, score)
    
    # Mark specific thresholds
    current_color = plt.gca().lines[-1].get_color()
    for t_val in [0.1, 0.3, 0.5, 0.7, 0.9]:
        # Find index of the threshold closest to t_val
        # Note: thresholds[0] is often max(score) + 1, so we filter it
        idx = np.argmin(np.abs(thresholds - t_val))
        
        plt.plot(fpr_vals[idx], tpr_vals[idx], 'o', color=current_color, markersize=6, mec='k')
        
        # Annotate for the RF model to avoid label overcrowding
        if name == "RF":
            plt.annotate(f"Thr={t_val}", (fpr_vals[idx], tpr_vals[idx]), 
                         xytext=(10, -5), textcoords='offset points', 
                         fontsize=9, fontweight='bold')

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.7, label="Random")

plt.xlabel("False Positive Rate (Contamination)")
plt.ylabel("True Positive Rate (Completeness)")
plt.title("ROC Curve on Validation Set: Quasar Identification")
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Precision–Recall Curve for Quasar Classification

Precision–Recall curves are more informative than ROC curves for rare-object searches, showing the trade-off between correctly identifying quasars and avoiding false positives.

### Why Precision–Recall is More Informative Than ROC

Precision–Recall focuses on:

- **Precision**: How many predicted quasars are actually quasars?
- **Recall**: How many real quasars did we recover?

This is directly aligned with survey goals:

- High precision → efficient use of telescope time
- High recall → completeness of scientific sample

For rare objects like quasars:

> A model with good ROC performance can still have poor precision.

This is why Precision–Recall curves and Average Precision (AP)
are the preferred metrics for rare object detection.

Even a model that predicts many false positives can maintain a low False Positive Rate
simply because the number of non-quasars is very large.

This is why ROC curves can overestimate performance in imbalanced settings.

### Final Insight on Imbalance

In astronomical classification:

- Class imbalance is the rule, not the exception
- Rare objects are often the most scientifically valuable
- Evaluation must reflect the scientific objective

For this reason:
- Precision–Recall curves are preferred over ROC
- Calibrated probabilities are essential for ranking candidates


In [ ]:
# Baseline = fraction of positives
baseline_val = qso_true_val.mean()
print("Quasar fraction (Random baseline):", baseline_val)

plt.figure(figsize=(8, 6))

for name, score in [("Dummy", qso_dummy_val), ("Logistic", qso_logit_val), ("RF", qso_rf_val)]:
    
    p, r, t = precision_recall_curve(qso_true_val, score)
    ap = average_precision_score(qso_true_val, score)
    
    # Plot the main curve
    line, = plt.plot(r, p, label=f"{name} AP={ap:.3f}", lw=2)
    current_color = line.get_color()

    # Mark specific thresholds (e.g., 0.2, 0.5, 0.8)
    # Note: len(t) is len(p) - 1, so we slice p and r accordingly
    for t_val in [0.1, 0.3, 0.5, 0.7, 0.9]:
        # Find the index of the threshold closest to t_val
        idx = np.argmin(np.abs(t - t_val))
        
        plt.plot(r[idx], p[idx], 'o', color=current_color, markersize=6, mec='k')
        
        # Only annotate thresholds for the RF to keep the plot clean, 
        # or for all if you prefer.
        if name == "RF":
            plt.annotate(f"Thr={t_val}", (r[idx], p[idx]), xytext=(5, 5), 
                         textcoords='offset points', fontsize=8, fontweight='bold')

plt.axhline(baseline_val, linestyle="--", color="gray", alpha=0.7,
            label=f"Random baseline ({baseline_val:.3f})")

plt.xlabel("Recall (Completeness)")
plt.ylabel("Precision (Purity)")
plt.title("Precision–Recall curve on Validation Set (Quasar)")
plt.legend(loc='upper left')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
pred_rf_test = rf.predict(X_test)
proba_rf_test = rf.predict_proba(X_test)

print("Classification Report on Test Set: RF")
print(classification_report(y_test, pred_rf_test, target_names=class_names))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_rf_test))


## Physical Models and Explainability

### Logistic Regression as a Physical Model

Logistic regression learns smooth decision boundaries in color space.

This is conceptually similar to classical astronomy selection cuts:

- Quasars: UV excess → low (u−g)
- Galaxies: redder colors
- Stars: follow stellar locus

The model generalizes these ideas into probabilistic form:
instead of hard cuts, we obtain continuous likelihoods.

**Coefficients**:
- are shown in standardized feature space and should be interpreted comparatively, not as direct physical sensitivities.
- indicate how changes in standardized features shift the relative log-odds between classes.

In [ ]:
coef = logit.named_steps["clf"].coef_

coef_df = pd.DataFrame(
    coef,
    index=class_names,
    columns=logit.named_steps["features"].get_feature_names_out(X_test.columns),
)

plt.figure(figsize=(12, 6))

im = plt.imshow(coef_df, aspect="auto", cmap="coolwarm")

plt.yticks(range(len(class_names)), class_names)
plt.xticks(range(len(logit.named_steps["features"].get_feature_names_out(X_test.columns))), 
           logit.named_steps["features"].get_feature_names_out(X_test.columns), rotation=90)

plt.colorbar(im, label="Logistic regression coefficient")

plt.title("Logistic regression coefficient per class")
plt.tight_layout()
plt.show()

In [ ]:
for i, cls in enumerate(class_names):

    coef_series = pd.Series(coef[i], index=logit.named_steps["features"].get_feature_names_out(X_test.columns))

    top = pd.concat([
        coef_series.sort_values().head(8),
        coef_series.sort_values(ascending=False).head(8)
    ])

    plt.figure(figsize=(10, 4))
    top.plot(kind="bar")

    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"{cls}: strongest positive and negative drivers")
    plt.ylabel("Logistic coefficient")
    plt.tight_layout()
    plt.show()

In [ ]:
# Logistic regression: probability vs color
# Build color space directly
u_g_range = np.linspace( X_train_color["u_g"].min(), X_train_color["u_g"].max(), 200)
g_r_fixed = np.median(X_train_color["g_r"])
X_range_color = pd.DataFrame({
    "u_g": u_g_range,
    "g_r": g_r_fixed
})

# Train a simple model in the SAME space (important!)
vis_model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, random_state=SEED, solver="lbfgs"))
])

vis_model.fit(X_train_color, y_train)

# Predict probabilities
probs = vis_model.predict_proba(X_range_color)

# Plot
plt.figure(figsize=(7,5))
for i, cls in enumerate(class_names):
    plt.plot(u_g_range, probs[:, i], label=cls)

plt.xlabel("u - g")
plt.ylabel("Predicted probability")
plt.title("Class probability vs u - g")
plt.legend()
plt.show()

### Random Forest feature importances

Notice that RF impurity importances are biased toward:
- high-cardinality features,
- noisy continuous variables,
- correlated predictors.

Be careful not to over-interpret RF feature importances physically, and consider SHAP.


In [ ]:
importances = rf.named_steps["clf"].feature_importances_

feat_importance = pd.Series(
    importances,
    index=rf.named_steps["features"].get_feature_names_out(X_test.columns)
).sort_values(ascending=False)

plt.figure(figsize=(10,6))
feat_importance.head(20).plot(kind="bar")
plt.title("Top Feature Importances")
plt.tight_layout()
plt.show()

# SHAP Explainability

SHAP values quantify how each feature contributes to a model prediction.

They allow:
- global feature importance analysis
- local (single-object) explanations

In [ ]:
# Use small sample for speed
# You are calculating the "Global Feature Importance" for your Random Forest.

X_shap = X_test.sample(min(500, len(X_test)), random_state=SEED)
X_shap_fe = rf.named_steps["features"].transform(X_shap)

explainer = shap.TreeExplainer(rf.named_steps["clf"])
print("Generating SHAP summary plot...")


### Why SHAP?

SHAP provides a consistent way to interpret tree-based models by attributing prediction changes to input features.

It is used here to verify whether the model learns physically meaningful structure rather than spurious correlations.


- Use Random Forest Importance as a fast diagnostic during model development to prune irrelevant features or identify leakage.

- Use SHAP for the final scientific analysis. SHAP helps diagnose which observables the model relies on and can reveal physically suspicious behavior or survey artifacts.
  
>In highly correlated feature spaces, SHAP values distribute importance across correlated observables and should not be interpreted as uniquely causal.

In [ ]:
# Use a subset for speed

def get_shap_values(explainer, X):
    """
    Returns SHAP values in shape:
    (n_samples, n_features, n_classes)
    Works across SHAP versions.
    """
    sv = explainer.shap_values(X)

    # Old API: list of arrays (one per class)
    if isinstance(sv, list):
        return np.stack(sv, axis=-1)

    # New API: already in correct shape
    return sv

shap_values = get_shap_values(explainer, X_shap_fe)

print("SHAP values shape:", shap_values.shape)

### Global SHAP Interpretation

SHAP values show which features most strongly influence model predictions across the dataset.

Results highlight whether classification is driven by:
- photometric colors
- morphology
- or mixed feature interactions

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))

shap_importance = pd.Series(mean_abs_shap, 
            index=rf.named_steps["features"].get_feature_names_out(X_test.columns)).sort_values(ascending=False)

plt.figure(figsize=(10,6))
shap_importance.head(20).plot(kind="bar")

plt.title("Global Feature Importance (SHAP)")
plt.ylabel("Mean |SHAP value|")
plt.tight_layout()
plt.show()

### Physical Interpretation

For quasars, important signals typically include:
- UV excess (color features)
- point-source morphology
- weak galaxy-like size indicators

SHAP highlights whether the model relies on these expected astrophysical signatures.


For an astrophysicist, the **SHAP summary plot** is one of the most powerful tools for validating that the model relies on astrophysically plausible observables rather than obvious survey artifacts or leakage. It combines feature importance with the directional impact of each variable.

To read the plot, you look for the correlation between **Color** and **X-axis position**.

* **Positive Correlation:** If high feature values (**Red dots**) tend to appear on the **positive SHAP side** (**on the right**), larger values of that feature tend to increase the model output.

* **Negative Correlation:** If you see **Blue dots on the right** and **Red dots on the left**, the relationship is inverse.

* **Overlapping Colors:** If Red and Blue dots are mixed together around zero, the feature has a non-linear or inconsistent impact on the prediction.


In [ ]:
sv_qso = shap_values[:, :, qso_idx]

shap.summary_plot(sv_qso, X_shap_fe)

In [ ]:
# QSO-specific feature importance
mean_abs_qso = np.abs(sv_qso).mean(axis=0)
qso_importance = pd.Series(mean_abs_qso, 
            index=rf.named_steps["features"].get_feature_names_out(X_test.columns)).sort_values(ascending=False)

print("Top features driving QSO classification:")
print(qso_importance.head(15))

# Local Explanation

SHAP explains individual predictions by decomposing the model output into feature-level contributions.

The explanation shows how each feature shifts the prediction relative to the model baseline.

For quasars, dominant signals typically include:
- UV-excess color signatures
- point-source morphology that distinguishes them from extended galaxies

In [ ]:
# Local explanation (single object)

# Find indices where true label AND prediction are both QSO
high_conf_qso_candidates = np.where((y_test == qso_idx) & (pred_rf_test == qso_idx))[0]

# Safety check: avoid crash if no such sample exists
if len(high_conf_qso_candidates) == 0:
    raise ValueError("No correctly predicted QSO found in test set.")

# Pick first candidate
high_conf_qso_idx = high_conf_qso_candidates[0]

# Select object
obj = X_test.iloc[[high_conf_qso_idx]]

# Predict again for this object (for consistency)
pred_obj = rf.predict(obj)[0]

print("True label:", class_names[y_test[high_conf_qso_idx]])
print("Predicted:", class_names[pred_obj])


### Interpreting the Local Explanation

The waterfall plot shows how the prediction is built:
- The baseline is the average model output

Each feature pushes the prediction:
- upward → toward the predicted class
- downward → away from it

### Astrophysical reading

For a quasar prediction, we typically expect:
- colors consistent with UV excess
- morphology consistent with a point source
- absence of strong galaxy-like size indicators

This allows us to translate a machine learning output into a physically interpretable explanation.

In [ ]:
obj_fe = rf.named_steps["features"].transform(obj)

# Compute SHAP values safely (must be defined elsewhere)
sv_one = get_shap_values(explainer, obj_fe)

# Extract SHAP values for predicted class
sv = sv_one[0, :, pred_obj]

# Base value (expected model output)
base = explainer.expected_value
if isinstance(base, (list, np.ndarray)):
    base = base[pred_obj]
    
# Build explanation object
exp = shap.Explanation(
    values=sv,
    base_values=base,
    data=obj_fe.iloc[0],
    feature_names=rf.named_steps["features"].get_feature_names_out(X_test.columns)
)

shap.plots.waterfall(exp)

## Probability Calibration

- Why Calibration Matters:
    - A model is well calibrated if predicted probabilities match observed frequencies.
    - This is essential for ranking astronomical objects for follow-up observations.
- Calibration Strategy:
    - Calibration is performed using cross-validated Platt scaling and isotonic regression.
    - This avoids using a separate validation set while reducing overfitting risk.
    
###### Probability Calibration of the Classifier

Two calibration methods are applied and compared:

- Sigmoid (Platt scaling), which fits a parametric sigmoid function to map raw model scores to calibrated probabilities
- Isotonic regression, which uses a non-parametric approach and can capture more flexible calibration curves

After fitting on the calibration cross-validation set, both calibrated models are used to generate probability estimates on the test set. 

These calibrated probabilities are typically better aligned with true likelihoods than raw model outputs.

In [ ]:
X_cal_train, X_cal_holdout, y_cal_train, y_cal_holdout = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train
)

qso_true_cal_holdout = (y_cal_holdout == qso_idx).astype(int)

print("Calibration Training size:", len(X_cal_train))
print("Calibration Holdout size:", len(X_cal_holdout))


In [ ]:
# Base and calibrated models

rf_params = dict(
    n_estimators=500,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)


rf_base = Pipeline([
        ("features", FeatureEngineer()),
        ("clf", RandomForestClassifier(**rf_params))
])

# Calibrated versions
rf_platt = Pipeline([
        ("features", FeatureEngineer()),
        ("clf", CalibratedClassifierCV(
            estimator=RandomForestClassifier(**rf_params),
            method="sigmoid",   # Platt scaling
            cv=5                # cross-validated calibration
        ))
])

rf_iso = Pipeline([
        ("features", FeatureEngineer()),
        ("clf", CalibratedClassifierCV(
            estimator=RandomForestClassifier(**rf_params),
            method="isotonic",  # flexible non-parametric calibration
            cv=5
        ))
])

# Fit models
rf_base.fit(X_cal_train,  y_cal_train)
rf_platt.fit(X_cal_train, y_cal_train)
rf_iso.fit(X_cal_train,   y_cal_train)

# Predict probabilities
proba_rf_base_cal_holdout  = rf_base.predict_proba(X_cal_holdout)
proba_rf_platt_cal_holdout = rf_platt.predict_proba(X_cal_holdout)
proba_rf_iso_cal_holdout   = rf_iso.predict_proba(X_cal_holdout)

## Confidence Distribution and the Impact of Calibration

This step examines how calibration affects the **confidence of model predictions**.

Instead of focusing only on which class is predicted, we analyze **how certain the model is** in its predictions.

- We evaluate model confidence using the maximum predicted class probability. This measures how decisive the model is for each prediction.
- Calibration affects how reliably objects can be prioritized for telescope follow-up. Overconfident models can waste observational resources.

We plot histograms of prediction confidence for each model.

- Each curve shows how many objects fall into a given confidence range  
- Step-style histograms are used to avoid visual overlap  

Two reference lines are added:

- **0.33 (random guessing)**  
  - no discriminative power  

- **0.95 ("science-grade" confidence)**  
  - typical threshold for high-priority follow-up targets  

Notice that a model can be perfectly calibrated but useless if it predicts $0.33$ for everything.

- Calibration primarily affects probability quality and often has limited impact on hard-label accuracy.

- The goal is improved probability reliability, not better hard-label accuracy.

Calibration is not just a statistical correction:

> it directly controls how many objects we trust enough to observe

A model that is slightly less confident—but better calibrated—can lead to:

- more efficient telescope time usage  
- higher purity in selected samples  
- more reliable scientific conclusions  


In [ ]:
# Random baseline (correct)
random_probs = rng.dirichlet(np.ones(len(class_names)), size=len(y_cal_holdout))
random_conf = random_probs.max(axis=1)

conf_data = pd.DataFrame({
    'Uncalibrated': proba_rf_base_cal_holdout.max(axis=1),
    'Platt': proba_rf_platt_cal_holdout.max(axis=1),
    'Isotonic': proba_rf_iso_cal_holdout.max(axis=1),
    'Random Classifier': random_conf
})

plt.figure(figsize=(10, 6))
colors = ['#e31a1c', '#33a02c', '#1f78b4', '#7f7f7f']
for i,col in enumerate(conf_data.columns):
    sns.histplot(conf_data[col], label=col, element="step", fill=False, bins=40, color=colors[i])

plt.axvline(0.95, color='orange', linestyle='--', label='Selection threshold')

plt.title("Prediction Confidence: Calibration vs Random Model")
plt.xlabel("Max class probability")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## Extracting Class-Specific Probabilities

We convert the multiclass classification problem into a binary perspective for a single class of interest, in this case quasars (QSO), and evaluate calibration in a one-vs-rest binary setting for the QSO class.

This enables evaluation of how calibration affects probability estimates for a specific scientific class of interest.

In [ ]:
# Focus on quasars (QSO)

qso_rf_base_cal_holdout  = proba_rf_base_cal_holdout[:, qso_idx]
qso_rf_platt_cal_holdout = proba_rf_platt_cal_holdout[:, qso_idx]
qso_rf_iso_cal_holdout   = proba_rf_iso_cal_holdout[:, qso_idx]

### From Classification to Decision-Making

In real surveys, we do not classify all objects—we select a small subset for follow-up.

This becomes a **resource allocation problem**:

- Telescope time is limited
- Observations are expensive
- We must maximize scientific return

Ranking by calibrated probability allows us to:
- prioritize high-confidence candidates
- maximize the expected number of true discoveries

This is why **probability calibration is more important than raw accuracy**.

In practice, selection may optimize expected scientific return rather than raw precision.


In [ ]:
# Telescope allocation simulation

N_obj = 100  # number of objects we can observe

# Rank by quasar probability
rank_idx_base_cal_holdout  = np.argsort(qso_rf_base_cal_holdout)[::-1]
rank_idx_iso_cal_holdout   = np.argsort(qso_rf_iso_cal_holdout)[::-1]
rank_idx_platt_cal_holdout = np.argsort(qso_rf_platt_cal_holdout)[::-1]

top_idx_base_cal_holdout  = rank_idx_base_cal_holdout[:N_obj]
top_idx_iso_cal_holdout   = rank_idx_iso_cal_holdout[:N_obj]
top_idx_platt_cal_holdout = rank_idx_platt_cal_holdout[:N_obj]

# True number of quasars recovered
n_qso_found_base_cal_holdout  = qso_true_cal_holdout[top_idx_base_cal_holdout].sum()
n_qso_found_iso_cal_holdout   = qso_true_cal_holdout[top_idx_iso_cal_holdout].sum()
n_qso_found_platt_cal_holdout = qso_true_cal_holdout[top_idx_platt_cal_holdout].sum()

print(f"Selected {N_obj} candidates")
print(f"Recovered QSOs Uncalibrated: {n_qso_found_base_cal_holdout}")
print(f"Recovered QSOs Isotonic: {n_qso_found_iso_cal_holdout}")
print(f"Recovered QSOs Platt: {n_qso_found_platt_cal_holdout}")
print(f"Precision Uncalibrated: {n_qso_found_base_cal_holdout / N_obj:.3f}")
print(f"Precision Isotonic: {n_qso_found_iso_cal_holdout / N_obj:.3f}")
print(f"Precision Platt: {n_qso_found_platt_cal_holdout / N_obj:.3f}")

In [ ]:
# Random baseline
rand_idx = rng.choice(len(qso_true_cal_holdout), N_obj, replace=False)
print("Random QSOs:", qso_true_cal_holdout[rand_idx].sum())

# Threshold strategy
threshold = 0.9
thresh_idx_base_cal_holdout  = np.where(qso_rf_base_cal_holdout  > threshold)[0][:N_obj]
thresh_idx_iso_cal_holdout   = np.where(qso_rf_iso_cal_holdout   > threshold)[0][:N_obj]
thresh_idx_platt_cal_holdout = np.where(qso_rf_platt_cal_holdout > threshold)[0][:N_obj]
print("Threshold QSOs Uncalibrated:", qso_true_cal_holdout[thresh_idx_base_cal_holdout].sum())
print("Threshold QSOs Isotonic:", qso_true_cal_holdout[thresh_idx_iso_cal_holdout].sum())
print("Threshold QSOs Platt:", qso_true_cal_holdout[thresh_idx_platt_cal_holdout].sum())

# Reliability Diagram for Model Calibration

Assess probability calibration by comparing predicted probabilities with observed frequencies.

A calibration curve is computed:

- by grouping predictions into bins and 
- comparing the average predicted probability with the observed frequency of the positive class in each bin. 

This plot answers: **"If the model is 80% sure, is it right 80% of the time?"**

* **The Diagonal (Dashed Line):** Represents **Perfect Calibration**. If your model sits here, the predicted probability equals the true physical frequency.
* **The X-axis:** The "confidence" the model feels.
* **The Y-axis:** The "reality" (fraction of actual quasars found in that bin).

### Why we "Calibrate" in Astronomy

1. **RF (Uncalibrated):** Tree ensembles often produce poorly calibrated probabilities, frequently exhibiting over- or under-confidence depending on depth, class imbalance, and dataset structure. Because trees rarely agree 100%, the probabilities tend to get pulled toward the center.
2. **Over-confidence:** Conversely, if the curve falls *below* the diagonal, the model is "arrogant"—it claims 90% certainty, but the sample is only 70% pure.
3. **Scientific Utility:** We use **Platt Scaling** or **Isotonic regression** to realign the model's output toward the diagonal. The goal is to make your published "Probability" column a more statistically meaningful tool for other researchers.
4. **The Big Caveat (Selection Bias):** Calibration is only reliable if your calibration sample is representative of the broader deployment population. In astronomy, strong spectroscopic selection effects often bias these samples, meaning even a "calibrated" model can still yield skewed probability estimates in the wild.

> Accuracy tells you if the model is smart; Calibration attempts to make the model's **uncertainty** more honest—though it is always bound by the limits of your training data. But calibration is not invariant under distribution shift.

In [ ]:
def plot_reliability(y_true, prob, label):
    frac_pos, mean_pred = calibration_curve(
        y_true, prob,
        n_bins=10,
        strategy="quantile"
    )
    plt.plot(mean_pred, frac_pos, marker="o", label=label)

plt.figure(figsize=(7,5))

plot_reliability(qso_true_cal_holdout, qso_rf_base_cal_holdout,  "RF (uncalibrated)")
plot_reliability(qso_true_cal_holdout, qso_rf_platt_cal_holdout, "Platt scaling")
plot_reliability(qso_true_cal_holdout, qso_rf_iso_cal_holdout,   "Isotonic")

# Perfect calibration line
plt.plot([0,1],[0,1],"--", color="black", label="Perfect calibration")

plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Reliability Diagram (Quasar)")
plt.legend()
plt.tight_layout()
plt.show()

# Model Evaluation Summary

Compare classification performance and probability quality across different models using standard metrics.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "log_loss": log_loss(y_test, proba)
    }



In [ ]:
results = pd.DataFrame([
    evaluate_model("RF", rf_base, X_cal_holdout, y_cal_holdout),
    evaluate_model("RF + Platt", rf_platt, X_cal_holdout, y_cal_holdout),
    evaluate_model("RF + Isotonic", rf_iso, X_cal_holdout, y_cal_holdout)
])

print(results.sort_values("log_loss"))

### Expected Calibration Error (ECE)

ECE quantifies the "honesty" of a model by measuring the gap between predicted confidence and observed frequency.

$$ \text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{N} \, \big| \text{acc}(B_m) - \text{conf}(B_m) \big| $$

While metrics like Accuracy or AUC measure how well a model *separates* classes, **ECE isolates calibration**.

It answers: "If a model predicts 80% confidence, does it actually get the answer right 80% of the time?"

Calibration is critical for **resource allocation**. When prioritizing targets for spectroscopic follow-up:

* **Miscalibrated probabilities** lead to inefficient telescope use and biased population statistics.
* **Well-calibrated probabilities** ensure that "80% confident" candidates represent a reliable success rate for survey planning.

> ECE is sensitive to binning strategies. For a complete diagnostic, always evaluate it alongside **Reliability Diagrams** and the **Brier Score**.

In [ ]:
def expected_calibration_error(y_true, prob, n_bins=10):
    """
    Computes Expected Calibration Error (ECE)

    Parameters:
    - y_true: binary labels (0/1)
    - prob: predicted probabilities for positive class
    - n_bins: number of bins

    Returns:
    - ece: scalar calibration error
    """

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    binids = np.digitize(prob, bins) - 1
    binids = np.clip(binids, 0, n_bins - 1)

    ece = 0.0
    N = len(y_true)

    for i in range(n_bins):
        idx = binids == i
        if np.sum(idx) == 0:
            continue

        bin_acc = y_true[idx].mean()
        bin_conf = prob[idx].mean()

        ece += (np.sum(idx) / N) * np.abs(bin_acc - bin_conf)

    return ece


In [ ]:
# Compute ECE for QSO classification
ece_rf_base = expected_calibration_error(qso_true_cal_holdout, qso_rf_base_cal_holdout)
ece_platt   = expected_calibration_error(qso_true_cal_holdout, qso_rf_platt_cal_holdout)
ece_iso     = expected_calibration_error(qso_true_cal_holdout, qso_rf_iso_cal_holdout)

print("Expected Calibration Error (QSO):")
print(f"RF raw:   {ece_rf_base:.4f}")
print(f"Platt:    {ece_platt:.4f}")
print(f"Isotonic: {ece_iso:.4f}")


### Brier Score: Quantifying Probability Quality

The Brier score measures the **Mean Squared Error** between predicted probabilities and true binary labels. 

It is a **strictly proper scoring rule**, meaning it is only minimized when the model predicts the true underlying probabilities.

$$ \text{Brier} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2 $$

The score rewards two distinct qualities simultaneously:

1. **Calibration:** Are the predicted probabilities accurate?
2. **Sharpness:** Are the predictions confidently close to 0 or 1?

In survey science, we use probabilities to rank targets for expensive spectroscopic follow-up. A lower Brier score directly translates to:

* **Scientific Efficiency:** Fewer "false alarms" and less wasted telescope time.
* **Reliable Ranking:** Higher confidence in quasar or transient selection.

> Because the Brier score mixes calibration and sharpness, we use **Expected Calibration Error (ECE)** to isolate whether the probabilities are honest regardless of their confidence.

In [ ]:
print("Brier scores (QSO):")
print("RF raw:   ", brier_score_loss(qso_true_cal_holdout, qso_rf_base_cal_holdout))
print("Platt:    ", brier_score_loss(qso_true_cal_holdout, qso_rf_platt_cal_holdout))
print("Isotonic: ", brier_score_loss(qso_true_cal_holdout, qso_rf_iso_cal_holdout))


## Why this matters scientifically

In real surveys we often rank candidates by probability.

Examples:

- choose top quasar candidates for spectroscopy
- prioritize rare transient hosts
- build clean galaxy samples

If probabilities are miscalibrated, telescope time is wasted.


## Final takeaway

In astronomy classification:

> ranking objects correctly is often more important than assigning a hard label

In survey astronomy:

* **Precision–Recall (PR) analysis** ensures target selection is scientifically efficient under class imbalance and limited follow-up resources.
* **Explainability (e.g. SHAP)** helps identify which observables drive model decisions, allowing astronomers to detect spurious correlations, survey artifacts, or physically implausible behavior.
* **Calibration** ensures predicted probabilities are scientifically meaningful.

Together, these tools help transform ML models from black-box predictors into scientifically interpretable inference systems.

However, notice that a major challenge in survey ML is that:
- Spectroscopic training sets are not representative samples of the full photometric population. 
- This induces selection bias and can invalidate both classification metrics and probability calibration.
